In [50]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import requests
import pandas as pd
from datetime import timedelta

StatementMeta(, 07fd4f81-1eaa-4e54-900e-74f680c3c9e1, 52, Finished, Available, Finished, False)

In [51]:
base_url = "https://raw.githubusercontent.com/nandorsilva/azure-fabric/refs/heads/main/datasets/"
bronze_table_path="abfss://88d42a16-2c0a-495a-97db-34b2c41a2439@onelake.dfs.fabric.microsoft.com/10aa08da-e1b2-4c37-9689-6d5e7643f00c/Tables/dbo/produtos"
default_start_date  = "20260131"

StatementMeta(, 07fd4f81-1eaa-4e54-900e-74f680c3c9e1, 53, Finished, Available, Finished, False)

In [ ]:
# Load existing data and convert to Pandas


try:
     # Verifica se é uma tabela Delta válida
    from delta.tables import DeltaTable

    if DeltaTable.isDeltaTable(spark, bronze_table_path):
        
        df_spark = spark.read.format("delta").load(bronze_table_path)

        if df_spark.count() > 0:
          df_pandas = df_spark.select("date").toPandas()

           
          most_recent_date = pd.to_datetime(
                df_pandas["date"], 
                format="%Y%m%d",
                errors="coerce"
            ).max()
        else:
            most_recent_date = pd.to_datetime(default_start_date, format="%Y%m%d")
    else:
        most_recent_date = pd.to_datetime(default_start_date, format="%Y%m%d")
except Exception:
    most_recent_date = pd.to_datetime(default_start_date, format="%Y%m%d")

# Calcula próxima data
next_date = (most_recent_date + timedelta(days=1)).strftime("%Y%m%d")


In [ ]:
# Download and load new data in a Pandas DataFrame
file_url = f"{base_url}{next_date}_produto.csv"
df_pandas_new = pd.read_csv(file_url)
df_pandas_new['date'] = pd.to_datetime(df_pandas_new['date'], errors="coerce")
df_pandas_new = df_pandas_new.sort_values(by="date", ascending=False)


In [54]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)
from delta.tables import DeltaTable

# 1) Defina o "contrato" do Bronze (ajuste para seus campos reais)
bronze_schema = StructType([
    StructField("produto_id", IntegerType(), True),
    StructField("date", DateType(), True),      # ou DateType() se você já vier como data
    StructField("time", StringType(), True),
    StructField("nome_produto", StringType(), True),
    StructField("valor", DoubleType(), True),
    StructField("categoria", StringType(), True),
    StructField("status", StringType(), True),
    StructField("departamento", StringType(), True),
])


StatementMeta(, 07fd4f81-1eaa-4e54-900e-74f680c3c9e1, 56, Finished, Available, Finished, False)

In [55]:
# Convert to Spark DataFrame and append in  table
df_spark_new = spark.createDataFrame(df_pandas_new, schema = bronze_schema)


if not DeltaTable.isDeltaTable(spark, bronze_table_path):
    df_spark_new.write.format("delta").mode("overwrite").save(bronze_table_path)
else:
    df_spark_new.write.format("delta").mode("append").save(bronze_table_path)



StatementMeta(, 07fd4f81-1eaa-4e54-900e-74f680c3c9e1, 57, Finished, Available, Finished, False)